# PPO Clean Implementation from Scratch

- PPO: [J. Schulman et al., "Proximal Policy Optimization Algorithms." arXiv preprint arXiv:1707.06347, 2017.](https://arxiv.org/abs/1707.06347.pdf)
- TRPO: [Schulman, John, et al. "Trust region policy optimization." International conference on machine learning. 2015.](http://proceedings.mlr.press/v37/schulman15.pdf)

There are two kinds of algorithms of PPO: PPO-Penalty and PPO-Clip. Here, we'll implement PPO-clip version.

TRPO computes the gradients with a complex second-order method. On the other hand, PPO tries to solve the problem with a first-order methods that keep new policies close to old. To simplify the surrogate objective, let $r(\theta)$ denote the probability ratio

$$ L^{CPI}(\theta) = \hat {\mathbb{E}}_t \left [ {\pi_\theta(a_t|s_t) \over \pi_{\theta_{old}}(a_t|s_t)} \hat A_t\right] = \hat {\mathbb{E}}_t \left [ r_t(\theta) \hat A_t \right ].$$

The objective is penalized further away from $r_t(\theta)$

$$ L^{CLIP}(\theta)=\hat {\mathbb{E}}_t \left [ \min(r_t(\theta) \hat A_t, \text{clip}(r_t(\theta), 1-\epsilon, 1+\epsilon)\hat A_t) \right ] $$

If the advantage is positive, the objective will increase. As a result, the action becomes more likely. If advantage is negative, the objective will decrease. AS a result, the action becomes less likely.

In [1]:
from collections import namedtuple, deque
from typing import List
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import gymnasium as gym


## Rollout Buffer

In [2]:
# ----------------------------
# Rollout buffer (on-policy)
# ----------------------------
Transition = namedtuple(
    "Transition",
    (
        "state",      # torch.FloatTensor [1, obs_dim]
        "action",     # torch.LongTensor  [1] (discrete action index)
        "reward",     # torch.FloatTensor [[1,1]]
        "done",       # torch.FloatTensor [[1,1]] (0. or 1.)
        "value",      # torch.FloatTensor [[1,1]]  (old V(s_t), detached)
        "log_prob",   # torch.FloatTensor [[1]]    (old log π(a|s), detached)
        "next_state", # torch.FloatTensor [1, obs_dim]
    )
)

class RolloutBuffer:
    def __init__(self, capacity: int):
        self.memory = deque(maxlen=capacity)
    def push(self, *args):
        self.memory.append(Transition(*args))
    def __len__(self):
        return len(self.memory)
    def clear(self):
        self.memory.clear()
    def as_list(self) -> List[Transition]:
        return list(self.memory)


## Network

In [3]:
# ----------------------------
# Hyperparameters
# ----------------------------
ENV_ID          = "CartPole-v1"
TOTAL_STEPS     = 100_000
ROLLOUT_STEPS   = 2048
GAMMA           = 0.99
LAMBDA          = 0.95
CLIP_EPS        = 0.2
UPDATE_EPOCHS   = 10
LR              = 3e-4
ENTROPY_COEF    = 0.01
VALUE_COEF      = 0.5
SEED            = 42
DEVICE          = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ----------------------------
# Actor-Critic for discrete actions
# ----------------------------
class ActorCritic(nn.Module):
    def __init__(self, obs_dim, act_dim, hidden=64):
        super().__init__()
        self.policy = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
            nn.Linear(hidden, act_dim)
        )
        self.value = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
            nn.Linear(hidden, 1)
        )

    def forward(self, obs):
        logits = self.policy(obs)             # [B, A]
        v = self.value(obs).squeeze(-1)       # [B]
        return logits, v

    def act(self, obs):
        """Sample action + old logp + old value (all detached for storage)."""
        logits, v = self.forward(obs)
        dist = torch.distributions.Categorical(logits=logits)
        a = dist.sample()                     # [B]
        logp = dist.log_prob(a)               # [B]
        return a.detach(), logp.detach(), v.detach()

    def logp_and_entropy(self, obs, actions):
        logits, _ = self.forward(obs)
        dist = torch.distributions.Categorical(logits=logits)
        logp = dist.log_prob(actions)         # [B]
        entropy = dist.entropy()              # [B]
        return logp, entropy



## GAE

In [4]:
# ----------------------------
# GAE(λ) returns
# ----------------------------
def compute_gae(rewards, dones, values, next_value, gamma=GAMMA, lam=LAMBDA):
    """
    rewards, dones, values: tensors shape [T]
    next_value: tensor scalar
    returns: tensor [T], advantages: tensor [T]
    """
    T = rewards.size(0)
    returns = torch.zeros(T, device=rewards.device)
    adv = torch.zeros(1, device=rewards.device)
    for t in reversed(range(T)):
        mask = 1.0 - dones[t]
        v_next = next_value if t == T - 1 else values[t + 1]
        delta = rewards[t] + gamma * v_next * mask - values[t]
        adv = delta + gamma * lam * mask * adv
        returns[t] = adv + values[t]
    advantages = returns - values
    return returns, advantages

## PPO Train

Algorithm:
    
    1. Perform 1 rollout (may have multiple episodes)
    2. Build tensors from the rollout (tensors for each state, action, reward, next_state, logp_old_policy)
    3. Compute GAE advantages and normalize it to mean 0 and var1 so that the learning is stable. (normalize actions in cont. action)
    4. PPO full batch update (may use mini batch+shuffling) for number of epochs: logp_old remains same; (logp and a) after each epoch changes.

In [5]:
# ----------------------------
# PPO training loop
# ----------------------------
def train():
    torch.manual_seed(SEED)
    np.random.seed(SEED)

    env = gym.make(ENV_ID)
    obs, _ = env.reset(seed=SEED)
    obs = torch.tensor(obs, dtype=torch.float32, device=DEVICE).unsqueeze(0)  # [1, obs]

    obs_dim = env.observation_space.shape[0]
    act_dim = env.action_space.n

    net = ActorCritic(obs_dim, act_dim).to(DEVICE)
    optim_ = optim.Adam(net.parameters(), lr=LR)

    rollout = RolloutBuffer(capacity=ROLLOUT_STEPS)

    total_steps = 0
    ep_return = 0.0
    returns_log = []

    while total_steps < TOTAL_STEPS:
        # --------- 1. Collect one rollout ---------
        rollout.clear()
        ## one rollout may has multiple episodes (depending on rollout_steps)
        for _ in range(ROLLOUT_STEPS):
            a_old, logp_old, v_old = net.act(obs)  # each shape [1]
            a_item = int(a_old.item())

            next_obs, reward, terminated, truncated, _ = env.step(a_item)
            done = float(terminated or truncated)

            next_obs_t = torch.tensor(next_obs, dtype=torch.float32, device=DEVICE).unsqueeze(0)
            r_t  = torch.tensor([[reward]], dtype=torch.float32, device=DEVICE)
            d_t  = torch.tensor([[done]],   dtype=torch.float32, device=DEVICE)
            a_t  = torch.tensor([a_item],   dtype=torch.long,    device=DEVICE)

            rollout.push(
                obs,           # state [1, obs]
                a_t,           # action [1]
                r_t,           # reward [[1,1]]
                d_t,           # done [[1,1]]
                v_old.unsqueeze(-1),          # value [[1,1]]
                logp_old.unsqueeze(-1),       # log_prob [[1]]
                next_obs_t     # next_state [1, obs]
            )

            ep_return += reward
            total_steps += 1
            obs = next_obs_t

            if done:
                returns_log.append(ep_return)
                ep_return = 0.0
                obs_np, _ = env.reset()
                obs = torch.tensor(obs_np, dtype=torch.float32, device=DEVICE).unsqueeze(0)

            if total_steps >= TOTAL_STEPS:
                break

        # --------- 2. Build tensors from rollout ---------
        traj = rollout.as_list()
        states     = torch.cat([t.state     for t in traj], dim=0)             # [T, obs]
        actions    = torch.cat([t.action    for t in traj], dim=0)             # [T]
        rewards    = torch.cat([t.reward    for t in traj], dim=0).squeeze(-1) # [T,1] -> [T]
        dones      = torch.cat([t.done      for t in traj], dim=0).squeeze(-1) # [T,1] -> [T]
        values_old = torch.cat([t.value     for t in traj], dim=0).squeeze(-1) # [T,1] -> [T]
        logp_old   = torch.cat([t.log_prob  for t in traj], dim=0).squeeze(-1) # [T,1] -> [T]

        # Bootstrap value from the last next_state
        with torch.no_grad():
            last_next_state = traj[-1].next_state
            _, v_last = net.forward(last_next_state)
            v_last = v_last.squeeze(0)  # scalar tensor
       # --------- 3. Compute GAE ---------
        with torch.no_grad():
            returns, advantages = compute_gae(rewards, dones, values_old, v_last)
            advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

        # --------- 4. PPO update (full-batch) ---------
        for _ in range(UPDATE_EPOCHS):
            logp_new, entropy = net.logp_and_entropy(states, actions)
            ratio = (logp_new - logp_old).exp()

            surr1 = ratio * advantages
            surr2 = torch.clamp(ratio, 1.0 - CLIP_EPS, 1.0 + CLIP_EPS) * advantages
            policy_loss = -torch.min(surr1, surr2).mean()

            _, values_pred = net.forward(states)
            value_loss = nn.functional.mse_loss(values_pred, returns)

            loss = policy_loss + VALUE_COEF * value_loss - ENTROPY_COEF * entropy.mean()

            optim_.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(net.parameters(), max_norm=0.5)
            optim_.step()

        if len(returns_log) and len(returns_log) % 10 == 0:
            print(f"Steps {total_steps:7d} | Avg return (last 10): {np.mean(returns_log[-10:]):.1f}")

    env.close()
    print("Finished. Avg return (last 100):", np.mean(returns_log[-100:]))
    return net, env


In [6]:
net, env = train()

Steps    6144 | Avg return (last 10): 23.2
Steps   18432 | Avg return (last 10): 63.4
Finished. Avg return (last 100): 226.43


## Evaluate

In [7]:
def eval_policy(net, env, episodes: int = 5, device: torch.device = DEVICE, render: bool = False) -> float:
    """
    Greedy evaluation for discrete-action ActorCritic:
    - Chooses argmax over policy logits (deterministic)
    - Returns average episodic return over `episodes`
    """
    net.eval()
    avg_return = 0.0

    with torch.no_grad():
        for _ in range(episodes):
            obs, _ = env.reset()
            s = torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
            done = False
            ep_ret = 0.0

            while not done:
                logits, _ = net.forward(s)                  # [1, A]
                a = torch.argmax(logits, dim=1).item()      # greedy action
                obs, r, terminated, truncated, _ = env.step(a)
                ep_ret += float(r)
                done = terminated or truncated
                if render:
                    env.render()
                if not done:
                    s = torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)

            avg_return += ep_ret

    net.train()  # keep training behavior consistent; remove if undesired
    return avg_return / episodes

# Example usage (with your PPO setup):
avg_ret = eval_policy(net, env, episodes=10)
print(f"Average return over 10 eval episodes: {avg_ret:.2f}")


Average return over 10 eval episodes: 410.80
